In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "kvu"
CODAPATH = Path("/kaggle/working/codapath")
if (CODAPATH / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(CODAPATH), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(CODAPATH), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(CODAPATH), "pull", "--ff-only", "origin", REPO_BRANCH])
elif CODAPATH.exists():
    raise RuntimeError(f"{CODAPATH} exists but is not a Git repository")
else:
    subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(CODAPATH)])
actual_branch = subprocess.check_output(["git", "-C", str(CODAPATH), "branch", "--show-current"], text=True).strip()
assert actual_branch == REPO_BRANCH, (actual_branch, REPO_BRANCH)
print("repo:", CODAPATH, "| branch:", actual_branch)

In [ ]:
%cd /kaggle/working/codapath
CODAPATH = "/kaggle/working/codapath"

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

In [ ]:
# ---- EDIT THIS CELL ----
# pathmnist | histoset | skintissue
DATASET = "pathmnist"

# random | coreset | typiclust | activeft | badge | entropy | margin
# codapath | scalpel | scalpel_multiscale | nucleus_al | nucleus_coverage
# uncertainty_herding | tcm | dropquery | refine | graph_deuce
SAMPLER_NAME = "nucleus_coverage"
SEED = 42

# One entry per full budget sweep, run back to back in THIS session so the
# dataset and the DINOv2 feature cache are loaded once instead of once per
# Kaggle session. Use [{}] for a sampler that has no variants.
#
# nucleus_al controlled variants:
# 1) {"cell_source": "crop_dino", "uncertainty_mode": "cell_margin"}
# 2) {"cell_source": "cellvit_embedding", "uncertainty_mode": "disagreement"}
# 3) {"cell_source": "cellvit_embedding", "uncertainty_mode": "fusion_concat"}
# 4) {"cell_source": "cellvit_embedding", "uncertainty_mode": "fusion_add"}
#
# nucleus_coverage controlled variants (the 3 experiments of DESIGN.md 4.3).
# Optional ablation: add {"coverage_source": ..., "missing_impute": "zero"} —
# it writes to a separate `_zero` run name, so it never overwrites these.
#
# graph_deuce controlled variants (EXPERIMENT.md Hướng 3 mục 7.10 — 2
# independent MLPVAE + DEUCE dual-neighbor-graph merge, 4 acquisition
# formulas sharing the same VAE+graph pipeline). VAE+graph is now cached
# across every variant/budget in THIS session (trained once, chat 2026-08-17
# GPU/speed fixes) — the first variant run pays for it, the rest reuse it.
# 1) {"acquisition_variant": "laplace_margin"}             # cheapest, SARGraphAL-faithful baseline
# 2) {"acquisition_variant": "uherding_swap_uncertainty"}  # dense DINO kernel unchanged, uncertainty -> laplace_margin
# 3) {"acquisition_variant": "uherding_swap_coverage"}     # LinearProbe+ECE uncertainty unchanged, coverage -> W_dual
# 4) {"acquisition_variant": "laplace_plus_ppr"}           # + Personalized PageRank coverage, weighted-sum blend
# Optional add-on for (1)/(4) only: {"per_point": True} re-solves Laplace
# learning after EVERY point picked (matches SARGraphAL's own sequential AL
# loop exactly) instead of once per round — ~5x more CG solves, writes to a
# separate `_perpoint` run name so it never overwrites the round-based run.
#
# scalpel_multiscale controlled variants (CLAUDE.md "SCALPEL-Multiscale — Key
# Facts"). The 3 toggles are independent; crop_mode is part of the x2/x4
# feature-cache filename, so switching it re-extracts rather than silently
# reusing another crop strategy's cache.
# 1) {"fusion_mode": "disagreement"}                          # Ver 1, BALD across zooms
# 2) {"fusion_mode": "concat"}                                # Ver 2, one probe on fused feats
# 3) {"fusion_mode": "disagreement", "coverage_space": "combined"}
# 4) {"fusion_mode": "disagreement", "crop_mode": "informative"}
SAMPLER_VARIANTS = [
    {"coverage_source": "dino"},
    {"coverage_source": "cellvit"},
    {"coverage_source": "concat"},
]

# Leave RUN_NAME=None to derive a collision-safe name per variant. A fixed
# RUN_NAME is only valid for a single variant (otherwise runs overwrite).
RUN_NAME = None

# Both cache paths below are only STARTING POINTS. Publishing /kaggle/working/<name>
# as a Kaggle Dataset remounts it one level deeper (.../<name>/<name>), so the next
# cell searches these paths and the attached inputs for the real cache directory
# and reports what it found. Set them to None to search from scratch.
FEATURE_DIR = "/kaggle/input/datasets/cryandrrich/nckh2026/features"
NUCLEUS_FEATURE_DIR = "/kaggle/input/datasets/cryandrrich/nckh2026/nucleus_features/nucleus_features"
OUTPUT_DIR = "/kaggle/working/checkpoints"

# Optional: override config.yaml's shared cumulative_budget for a scoped run
# without touching the global default every other sampler/experiment uses.
# Leave None to use the agreed protocol [25..200] straight from config.yaml.
# e.g. CUMULATIVE_BUDGET_OVERRIDE = [25, 50, 75, 100]
CUMULATIVE_BUDGET_OVERRIDE = None


In [ ]:
import os
from huggingface_hub import snapshot_download

# DINOv2 is public: no placeholder login/token is required.
# Enable Kaggle Internet, or set the model path to a mounted local snapshot.
print("Downloading facebook/dinov2-base...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if CODAPATH not in sys.path:
    sys.path.append(CODAPATH)

In [ ]:
import yaml
import torch

from run import main

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/cryandrrich/nckh2026"),
    Path("/kaggle/input/nckh2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), DATA_ROOT_CANDIDATES[0])
PATHMNIST_PATH  = str(DATA_ROOT / "pathmnist_224.npz")
HISTOSET_PATH   = str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14")
SKINTISSUE_PATH = str(DATA_ROOT / "SkinTissue/SkinTissue/tiles")

DATA_DICT = {
    "pathmnist":  PATHMNIST_PATH,
    "histoset":   HISTOSET_PATH,
    "skintissue": SKINTISSUE_PATH,
}

In [ ]:
CONFIG_PATH = "config/config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

assert DATASET in DATA_DICT, DATASET
data_path = Path(DATA_DICT[DATASET])
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
CUMULATIVE_BUDGET = CUMULATIVE_BUDGET_OVERRIDE or config["cumulative_budget"]
if CUMULATIVE_BUDGET_OVERRIDE is None:
    assert CUMULATIVE_BUDGET == [25, 50, 75, 100, 125, 150, 175, 200], CUMULATIVE_BUDGET
else:
    print("[budget] override in use:", CUMULATIVE_BUDGET)
assert torch.cuda.is_available(), "Attach a Kaggle GPU before running AL"
assert SAMPLER_VARIANTS, "SAMPLER_VARIANTS must hold at least one override dict (use [{}])"
assert RUN_NAME is None or len(SAMPLER_VARIANTS) == 1, (
    "A fixed RUN_NAME with several variants makes every run overwrite the previous one"
)

training_cfg = config.get("training", {})
dataset_info = config["datasets"][DATASET]
base_cfg = dict(config.get("samplers", {}).get(SAMPLER_NAME, {}))
sampler_cfgs = [{**base_cfg, **overrides} for overrides in SAMPLER_VARIANTS]
vit_name = config.get("models", {}).get("vit", "facebook/dinov2-base")
safe_vit = vit_name.replace("/", "_")

SEARCH_ROOTS = [DATA_ROOT, Path("/kaggle/input")]


def dir_containing(probe, hint=None, max_depth=3):
    """Return the directory D such that D/probe exists.

    Publishing /kaggle/working/<name> as a Kaggle Dataset remounts it as
    /kaggle/input/<slug>/<name>/<name> — one level deeper than the path anyone
    writes down — so a hard-coded cache path fails in a way that is tedious to
    debug from a stack trace. Search a few levels instead and print the hit.
    """
    probe = Path(probe)
    up = len(probe.parts) - 1
    roots = ([Path(hint)] if hint else []) + SEARCH_ROOTS
    for root in roots:
        if not root.exists():
            continue
        for depth in range(max_depth + 1):
            pattern = "/".join(["*"] * depth + list(probe.parts))
            for hit in sorted(root.glob(pattern)):
                return hit.parents[up]
    return None


# graph_deuce also reads the CellViT nucleus cache (VAE_cell + cell graph,
# EXPERIMENT.md Hướng 3 mục 7.1/7.4) — must be included here or
# NUCLEUS_FEATURE_DIR stays an unresolved placeholder and run.py's
# load_nucleus_cache fails deep inside the first variant instead of here.
if SAMPLER_NAME in ("nucleus_al", "nucleus_coverage", "graph_deuce"):
    nucleus_dir = dir_containing(f"{DATASET}_seed{SEED}/manifest.json", NUCLEUS_FEATURE_DIR)
    assert nucleus_dir is not None, (
        f"No nucleus cache found for {DATASET}_seed{SEED} under {NUCLEUS_FEATURE_DIR} "
        f"or {[str(r) for r in SEARCH_ROOTS]}. Attach the extraction dataset."
    )
    NUCLEUS_FEATURE_DIR = str(nucleus_dir)
    print("nucleus cache:", NUCLEUS_FEATURE_DIR)

# The DINOv2 cache is optional: run.py re-extracts on a miss. It must not
# re-extract into a read-only /kaggle/input though — that only fails AFTER the
# whole extraction, so fall back to a writable directory instead. Within one
# session the first variant pays the extraction and the other two reuse it.
found_feature_dir = dir_containing(f"{DATASET}_seed{SEED}_{safe_vit}_train.npy", FEATURE_DIR)
manifest_name = f"{DATASET}_seed{SEED}_{safe_vit}_manifest.json"
if found_feature_dir is None:
    print(f"[features] NOT FOUND for {DATASET}/seed{SEED}/{vit_name} — will extract this session.")
    for root in SEARCH_ROOTS:
        if root.exists():
            print("  attached under", root, "->", sorted(p.name for p in root.iterdir())[:20])
    FEATURE_DIR = "/kaggle/working/features"
elif not (found_feature_dir / manifest_name).is_file():
    print(f"[features] Found .npy at {found_feature_dir} but no {manifest_name}.")
    print("  A cache without a sample-order manifest is rejected by model.py "
          "(row/sample alignment cannot be verified) — extracting once this session.")
    FEATURE_DIR = "/kaggle/working/features"
else:
    FEATURE_DIR = str(found_feature_dir)
    print("features cache:", FEATURE_DIR)

# scalpel_multiscale writes its own x2/x4 caches through the SAME cache_dir as
# the base DINOv2 features (run.py passes feature_cache_dir to both). A missing
# scale cache under a read-only /kaggle/input therefore fails on np.save only
# AFTER the whole multi-scale extraction has run — check it up front instead.
if SAMPLER_NAME == "scalpel_multiscale":
    wanted = [
        f"{DATASET}_seed{SEED}_{safe_vit}_scale{s}_{cfg.get('crop_mode', 'center')}_train.npy"
        for cfg in sampler_cfgs for s in cfg.get("scale_factors", [2, 4])
    ]
    absent = [n for n in wanted if not (Path(FEATURE_DIR) / n).is_file()]
    if absent and str(FEATURE_DIR).startswith("/kaggle/input"):
        print("[multiscale] scale caches missing under a read-only input dir:", absent)
        print("  -> FEATURE_DIR switched to /kaggle/working/features "
              "(base DINOv2 features get re-extracted there too).")
        FEATURE_DIR = "/kaggle/working/features"

if not str(FEATURE_DIR).startswith("/kaggle/input"):
    Path(FEATURE_DIR).mkdir(parents=True, exist_ok=True)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
for cfg in sampler_cfgs:
    print(f"[{SAMPLER_NAME}] sampler_cfg =", cfg)
print("data:", data_path, "| output:", OUTPUT_DIR)

In [ ]:
import time

for index, sampler_cfg in enumerate(sampler_cfgs, start=1):
    started = time.time()
    print("=" * 70)
    print(f"VARIANT {index}/{len(sampler_cfgs)}: {SAMPLER_VARIANTS[index - 1]}")
    print("=" * 70)
    main(
        data_path=str(data_path),
        sampler_name=SAMPLER_NAME,
        num_classes=dataset_info["num_classes"],
        cumulative_budget=CUMULATIVE_BUDGET,
        data_descriptions=dataset_info["descriptions"],
        prompt_templates=config["prompt_templates"],
        sampler_cfg=sampler_cfg,
        probe_epochs=training_cfg["probe_epochs"],
        probe_lr=training_cfg["probe_lr"],
        device=torch.device(config["device"]),
        random_seed=SEED,
        save_dir=str(Path(OUTPUT_DIR) / DATASET),
        verbose=True,
        model_cfg=config.get("models", {}),
        feature_cache_dir=FEATURE_DIR,
        nucleus_cache_dir=NUCLEUS_FEATURE_DIR,
        run_name=RUN_NAME,
    )
    print(f"VARIANT {index} finished in {(time.time() - started) / 60:.1f} min")